In [ ]:
# GENERATED PORTFOLIOS DATASET
#import packages:
import pandas as pd
import datetime as dt
import math
import yfinance as yf
import numpy as np

url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies#Selected_changes_to_the_list_of_S&P_500_components'
data = pd.read_html(url) # scrape data for current S&P500 make up

sp500 = data[0]['Symbol'].to_list()


for component in sp500:
    check_for_nan = yf.download(component,'2023-07-01','2024-07-01')
    if check_for_nan.empty:
        sp500.remove(component)

print(len(sp500))



def create_bootstrap_datapoint():
# A function that takes no input and generates a list of
# 100 stocks from the current S&P500 with replacement, and gives
# two outputs, firstly the list of 100 stocks, then a list showing
# the distribution of sectors across the stock list

# 2023 to 2024
    
    
    sub_portfolio = []
    distribution = []
    technology = 0
    healthcare = 0
    fin_serv = 0
    cons_cyc = 0
    comm_serv = 0
    industrials = 0
    cons_def = 0
    energy = 0
    utilities = 0
    real_estate = 0
    basic_mat = 0
    
    while len(sub_portfolio) < 100:
        index = np.random.randint(0,len(sp500))
        ticker = str(sp500[index])
        stock = yf.Ticker(ticker)
        
        #data = stock.history(start='2023-07-01', end='2024-07-01')         # SPECIFIED TIME PERIOD:
        #if data.isnull().values.any():                                     # CHANGE DEPENDING ON THE
            #continue                                                       # PERIOD THE FINAL DATASET
                                                                           # SHOULD COVER
        sub_portfolio.append(ticker)
        
        sector = stock.info.get('sector', 'Sector information not available')
        
        if sector == 'Technology':
            technology += 1
        elif sector == 'Healthcare':
            healthcare += 1
        elif sector == 'Financial Services':
            fin_serv += 1
        elif sector == 'Consumer Cyclical':
            cons_cyc += 1
        elif sector == 'Communication Services':
            comm_serv += 1
        elif sector == 'Industrials':
            industrials += 1
        elif sector == 'Consumer Defensive':
            cons_def += 1
        elif sector == 'Energy':
            energy += 1
        elif sector == 'Utilities':
            utilities += 1
        elif sector == 'Real Estate':
            real_estate += 1
        elif sector == 'Basic Materials':
            basic_mat += 1
        else:
            print("Unknown sector")
            print(sector)
    
    distribution.append(technology)
    distribution.append(healthcare)
    distribution.append(fin_serv)
    distribution.append(cons_cyc)
    distribution.append(comm_serv)
    distribution.append(industrials)
    distribution.append(cons_def)
    distribution.append(energy)
    distribution.append(utilities)
    distribution.append(real_estate)
    distribution.append(basic_mat)
            
    
    return sub_portfolio, distribution



def calculate_index_volatility(index,start,end):
# Calculates the daily return for a generated list of stocks,
# takes as input the stock list, and start and end dates
# in the form 'YYYY-MM-DD'
    
    stock_info = yf.download(index[0],start,end)
    stock_info['Return'] = (stock_info['Close'] - stock_info['Open']) / stock_info['Open'] * 100
    
    index_returns = stock_info['Return'].tolist()
    
    for i in range(1,len(index)):
        stock_info = yf.download(index[i],start,end)
        stock_info['Return'] = (stock_info['Close'] - stock_info['Open']) / stock_info['Open'] * 100
        
        index_returns = [x + y for x, y in zip(index_returns, stock_info['Return'].tolist())]
    
    index_returns = [x/len(index) for x in index_returns]
    
    volatility = np.std(index_returns)
    
    return volatility
        



def calculate_price_change(index,start,end):
    index_open = 0
    index_close = 0
    
    for stock in index:
        stock_info = yf.download(stock,start,end)
        
        index_open += stock_info['Open'][0]
        index_close += stock_info['Close'][-1]
        
    price_change = (index_close - index_open) / index_open * 100
    
    return price_change
    

In [ ]:
# FUNCTION TO CREATE A DATAFRAME SHOWING SECTOR DISTRIBUTION
# AND BOTH PRICE CHANGE AND VOLATILITY FOR A SPECIFIED NUMBER
# OF DATA POINTS

# DATES IN THIS FUNCTION NEED TO BE CHANGED DEPENDING ON
# THE TIME PERIOD WANTED FOR THE DATASET


def create_dataset(n):
    
    technology = []
    healthcare = []
    fin_serv = []
    cons_cyc = []
    comm_serv = []
    industrials = []
    cons_def = []
    energy = []
    utilities = []
    real_estate = []
    basic_mat = []
    
    price_change = []
    volatility = []
    
    while len(technology) < n:
        
        try:
            index_i, distribution_i = create_bootstrap_datapoint()

            technology.append(distribution_i[0])
            healthcare.append(distribution_i[1])
            fin_serv.append(distribution_i[2])
            cons_cyc.append(distribution_i[3])
            comm_serv.append(distribution_i[4])
            industrials.append(distribution_i[5])
            cons_def.append(distribution_i[6])
            energy.append(distribution_i[7])
            utilities.append(distribution_i[8])
            real_estate.append(distribution_i[9])
            basic_mat.append(distribution_i[10])
            
            price_change_i = calculate_price_change(index_i,'2023-07-01','2024-07-01')
            volatility_i = calculate_index_volatility(index_i,'2023-07-01','2024-07-01')
            
            price_change.append(float(price_change_i))
            volatility.append(float(volatility_i))
            print(len(technology))
        except:
            continue
    
    print(price_change)
    print(volatility)
    
    data_table = {
        'Technology': technology,
        'Healthcare': healthcare,
        'Financial Services': fin_serv,
        'Consumer Cyclical': cons_cyc,
        'Communication Services': comm_serv,
        'Industrials': industrials,
        'Consumer Defensive': cons_def,
        'Energy': energy,
        'Utilities': utilities,
        'Real Estate': real_estate,
        'Basic Materials': basic_mat,
        'Return': price_change,
        'Volatility': volatility
    }
    
    dataframe = pd.DataFrame(data_table)
    
    return dataframe
        
    

In [ ]:
sp500_2024 = create_dataset(250)

In [ ]:
sp500_2024.to_csv('../data/sp500-2024.csv')